In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
import glob
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, num_classes=5, img_channels=3):
        super(Generator, self).__init__()
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        
        self.init_size = 16
        self.label_embedding = nn.Embedding(num_classes, latent_dim)
        
        self.l1 = nn.Sequential(nn.Linear(latent_dim * 2, 128 * self.init_size ** 2))
        
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.1),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, img_channels, 3, stride=1, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        label_embed = self.label_embedding(labels)
        gen_input = torch.cat((label_embed, z), -1)
        
        out = self.l1(gen_input)
        out = out.view(out.shape[0], 128, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        return img

class Discriminator(nn.Module):
    def __init__(self, num_classes=5, img_channels=3):
        super(Discriminator, self).__init__()
        
        def discriminator_block(in_filters, out_filters, bn=True):
            block = [nn.Conv2d(in_filters, out_filters, 3, 2, 1),
                    nn.LeakyReLU(0.2, inplace=True),
                    nn.Dropout2d(0.3)]
            if bn:
                block.append(nn.BatchNorm2d(out_filters, 0.8))
            return block
        
        self.model = nn.Sequential(
            *discriminator_block(img_channels, 16, bn=False),
            *discriminator_block(16, 32),
            *discriminator_block(32, 64),
            *discriminator_block(64, 128),
        )
        
        ds_size = 64 // 2 ** 4
        self.adv_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, 1), nn.Sigmoid())
        self.aux_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, num_classes), nn.Softmax(dim=1))
    
    def forward(self, img):
        out = self.model(img)
        out = out.view(out.shape[0], -1)
        validity = self.adv_layer(out)
        label = self.aux_layer(out)
        return validity, label

model_path = r"C:\Users\Administrator\OneDrive\Desktop\Gan+Unet project\2_anomaly_detection.pth"
checkpoint = torch.load(model_path, map_location='cpu')

print("Available keys in your model file:")
for key in checkpoint.keys():
    print(f"  - {key}")

generator = Generator()
discriminator = Discriminator()

if 'generator' in checkpoint:
    generator.load_state_dict(checkpoint['generator'])
elif 'G' in checkpoint:
    generator.load_state_dict(checkpoint['G'])
    exit()

if 'discriminator' in checkpoint:
    discriminator.load_state_dict(checkpoint['discriminator']) 
elif 'D' in checkpoint:
    discriminator.load_state_dict(checkpoint['D'])
    exit()

generator.eval()
discriminator.eval()

image_path = r"C:\Users\Administrator\OneDrive\Desktop\Gan+Unet project\tzh_wear\adhesive\pic5_3_135.png"
image = Image.open(image_path).convert('RGB')
original_size = image.size

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

image_tensor = transform(image).unsqueeze(0)

anomaly_colors = {
            0: [0, 255, 0],   
            1: [255, 0, 0],   
            2: [0, 0, 255],   
            3: [255, 255, 0],  
            4: [255, 0, 255],  
        }

class_names = ['normal', 'abrasive', 'adhasive', 'cracks', 'delamination']

with torch.no_grad():
    validity, class_probs = discriminator(image_tensor)
    predicted_class = torch.argmax(class_probs, dim=1).item()
    confidence = class_probs[0, predicted_class].item()
    
    z = torch.randn(1, 100)
    label_tensor = torch.tensor([predicted_class])
    reconstructed = generator(z, label_tensor)
    
    difference = torch.abs(image_tensor - reconstructed)
    anomaly_map = difference.mean(dim=1, keepdim=True)
    anomaly_mask = (anomaly_map > 0.5).float()
    
    anomaly_mask_np = anomaly_mask.squeeze().numpy()
    anomaly_mask_resized = cv2.resize(anomaly_mask_np, original_size)

original_image = cv2.imread(image_path)
original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

color_mask = np.zeros_like(original_image_rgb)
color_mask[anomaly_mask_resized > 0] = anomaly_colors[predicted_class]

result_image = cv2.addWeighted(original_image_rgb, 0.7, color_mask, 0.3, 0)

plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(original_image_rgb)
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 4, 2)
diff_visual = difference.squeeze().mean(axis=0).numpy()
plt.imshow(diff_visual, cmap='hot')
plt.title('Difference Map')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(anomaly_mask_resized, cmap='gray')
plt.title('Anomaly Mask')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(result_image)
plt.title(f'Class: {class_names[predicted_class]}\nConfidence: {confidence:.3f}')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Predicted wear type: {class_names[predicted_class]}")
print(f"Confidence: {confidence:.3f}")
print("\nClass probabilities:")
for i, class_name in enumerate(class_names):
    prob = class_probs[0, i].item()
    print(f"  {class_name}: {prob:.4f}")

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, num_classes=5, img_channels=3):
        super(Generator, self).__init__()
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        
        self.init_size = 16
        self.label_embedding = nn.Embedding(num_classes, latent_dim)
        
        self.l1 = nn.Sequential(nn.Linear(latent_dim * 2, 128 * self.init_size ** 2))
        
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.1),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, img_channels, 3, stride=1, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        label_embed = self.label_embedding(labels)
        gen_input = torch.cat((label_embed, z), -1)
        
        out = self.l1(gen_input)
        out = out.view(out.shape[0], 128, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        return img

class Discriminator(nn.Module):
    def __init__(self, num_classes=5, img_channels=3):
        super(Discriminator, self).__init__()
        
        def discriminator_block(in_filters, out_filters, bn=True):
            block = [nn.Conv2d(in_filters, out_filters, 3, 2, 1),
                    nn.LeakyReLU(0.2, inplace=True),
                    nn.Dropout2d(0.3)]
            if bn:
                block.append(nn.BatchNorm2d(out_filters, 0.8))
            return block
        
        self.model = nn.Sequential(
            *discriminator_block(img_channels, 16, bn=False),
            *discriminator_block(16, 32),
            *discriminator_block(32, 64),
            *discriminator_block(64, 128),
        )
        
        ds_size = 64 // 2 ** 4
        self.adv_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, 1), nn.Sigmoid())
        self.aux_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, num_classes), nn.Softmax(dim=1))
    
    def forward(self, img):
        out = self.model(img)
        out = out.view(out.shape[0], -1)
        validity = self.adv_layer(out)
        label = self.aux_layer(out)
        return validity, label

model_path = r"C:\Users\Administrator\OneDrive\Desktop\Gan+Unet project\2_anomaly_detection.pth"
checkpoint = torch.load(model_path, map_location='cpu')

print("Available keys in your model file:")
for key in checkpoint.keys():
    print(f"  - {key}")

generator = Generator()
discriminator = Discriminator()

if 'generator' in checkpoint:
    generator.load_state_dict(checkpoint['generator'])
elif 'G' in checkpoint:
    generator.load_state_dict(checkpoint['G'])
    exit()

if 'discriminator' in checkpoint:
    discriminator.load_state_dict(checkpoint['discriminator']) 
elif 'D' in checkpoint:
    discriminator.load_state_dict(checkpoint['D'])
    exit()

generator.eval()
discriminator.eval()

print("Model loaded successfully!")

image_path = r"C:\Users\Administrator\OneDrive\Desktop\Gan+Unet project\tzh_wear\cracks\pic4_6_12.png"
image = Image.open(image_path).convert('RGB')
original_size = image.size

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

image_tensor = transform(image).unsqueeze(0)

anomaly_colors = {
            0: [0, 255, 0],    # Green - normal
            1: [255, 0, 0],    # Red
            2: [0, 0, 255],    # Blue  
            3: [255, 255, 0],  # Cyan
            4: [255, 0, 255],  # Purple
        }

class_names = ['normal', 'abrasive', 'adhasive', 'cracks', 'delamination']

with torch.no_grad():
    validity, class_probs = discriminator(image_tensor)
    predicted_class = torch.argmax(class_probs, dim=1).item()
    confidence = class_probs[0, predicted_class].item()
    
    z = torch.randn(1, 100)
    label_tensor = torch.tensor([predicted_class])
    reconstructed = generator(z, label_tensor)
    
    difference = torch.abs(image_tensor - reconstructed)
    anomaly_map = difference.mean(dim=1, keepdim=True)
    anomaly_mask = (anomaly_map > 0.5).float()
    
    anomaly_mask_np = anomaly_mask.squeeze().numpy()
    anomaly_mask_resized = cv2.resize(anomaly_mask_np, original_size)

original_image = cv2.imread(image_path)
original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

color_mask = np.zeros_like(original_image_rgb)
color_mask[anomaly_mask_resized > 0] = anomaly_colors[predicted_class]

result_image = cv2.addWeighted(original_image_rgb, 0.7, color_mask, 0.3, 0)

plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(original_image_rgb)
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 4, 2)
diff_visual = difference.squeeze().mean(axis=0).numpy()
plt.imshow(diff_visual, cmap='hot')
plt.title('Difference Map')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(anomaly_mask_resized, cmap='gray')
plt.title('Anomaly Mask')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(result_image)
plt.title(f'Class: {class_names[predicted_class]}\nConfidence: {confidence:.3f}')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Predicted wear type: {class_names[predicted_class]}")
print(f"Confidence: {confidence:.3f}")
print("\nClass probabilities:")
for i, class_name in enumerate(class_names):
    prob = class_probs[0, i].item()
    print(f"  {class_name}: {prob:.4f}")

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn as nn
from torchvision import transforms

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, num_classes=5, img_channels=3):
        super(Generator, self).__init__()
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        self.init_size = 16
        self.label_embedding = nn.Embedding(num_classes, latent_dim)
        
        self.l1 = nn.Sequential(nn.Linear(latent_dim * 2, 128 * self.init_size ** 2))
        
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.1),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, img_channels, 3, stride=1, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        label_embed = self.label_embedding(labels)
        gen_input = torch.cat((label_embed, z), -1)
        out = self.l1(gen_input)
        out = out.view(out.shape[0], 128, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        return img

class Discriminator(nn.Module):
    def __init__(self, num_classes=5, img_channels=3):
        super(Discriminator, self).__init__()
        
        def discriminator_block(in_filters, out_filters, bn=True):
            block = [nn.Conv2d(in_filters, out_filters, 3, 2, 1),
                    nn.LeakyReLU(0.2, inplace=True),
                    nn.Dropout2d(0.3)]
            if bn:
                block.append(nn.BatchNorm2d(out_filters, 0.8))
            return block
        
        self.model = nn.Sequential(
            *discriminator_block(img_channels, 16, bn=False),
            *discriminator_block(16, 32),
            *discriminator_block(32, 64),
            *discriminator_block(64, 128),
        )
        
        ds_size = 64 // 2 ** 4
        self.adv_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, 1), nn.Sigmoid())
        self.aux_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, num_classes), nn.Softmax(dim=1))
    
    def forward(self, img):
        out = self.model(img)
        out = out.view(out.shape[0], -1)
        validity = self.adv_layer(out)
        label = self.aux_layer(out)
        return validity, label

In [ ]:
class DefectDetector:
    def __init__(self, model_path, img_size=64):
        self.img_size = img_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.generator = Generator(100, 5, 3).to(self.device)
        self.discriminator = Discriminator(5, 3).to(self.device)
        
        checkpoint = torch.load(model_path, map_location=self.device)
        self.generator.load_state_dict(checkpoint['generator'])
        self.discriminator.load_state_dict(checkpoint['discriminator'])
        
        self.generator.eval()
        self.discriminator.eval()
        
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)
        ])
        
        self.defect_colors = {
            0: [0, 150, 0],    # green
            1: [255, 0, 0],    # red - abrasive
            2: [0, 0, 255],    # blue - adhesive
            3: [255, 255, 0],  # yellow - cracks
            4: [255, 0, 255]   # pink - delamination
        }
        
        self.defect_names = {
            0: 'normal (green)',      
            1: 'abrasive (red)', 
            2: 'adhesive (blue)', 
            3: 'cracks (yellow)', 
            4: 'delamination (pink)'
        }

    def create_grid_pattern(self, tile_size, grid_size=4):
        """to make a grid"""
        pattern = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
        step = tile_size // grid_size
        
        for i in range(1, grid_size):
            y = i * step
            pattern[y-1:y+1, :, :] = self.defect_colors[0]
        
        for i in range(1, grid_size):
            x = i * step
            pattern[:, x-1:x+1, :] = self.defect_colors[0]
        return pattern

    def preprocess_image(self, image_path):
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Coudn't load image: {image_path}")
            
        original_h, original_w = image.shape[:2]
        target_size = 1200
        scale = target_size / max(original_h, original_w)
        new_w = int(original_w * scale)
        new_h = int(original_h * scale)
        
        resized_image = cv2.resize(image, (new_w, new_h))
        return resized_image

    def process_tiles(self, image, tile_size=128, stride=32):
        h, w = image.shape[:2]
        tiles = []
        positions = []
        
        for y in range(0, h - tile_size + 1, stride):
            for x in range(0, w - tile_size + 1, stride):
                tile = image[y:y + tile_size, x:x + tile_size]
                tiles.append(tile)
                positions.append((x, y, x + tile_size, y + tile_size))
        
        return tiles, positions

    def analyze_tile(self, tile):
        tile_rgb = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(tile_rgb)
        
        image_tensor = self.transform(pil_image).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            validity, class_probs = self.discriminator(image_tensor)
            predicted_class = torch.argmax(class_probs, dim=1).item()
            confidence = class_probs[0, predicted_class].item()
            
            if predicted_class == 0:
                return {
                    'class': predicted_class,
                    'confidence': confidence,
                    'anomaly_map': None,
                    'is_normal': True
                }
            elif predicted_class != 0 and confidence > 0.6:
                z = torch.randn(1, 100).to(self.device)
                label_tensor = torch.tensor([predicted_class], device=self.device)
                reconstructed = self.generator(z, label_tensor)
                
                difference = torch.abs(image_tensor - reconstructed)
                anomaly_map = difference.mean(dim=1, keepdim=True)
                
                return {
                    'class': predicted_class,
                    'confidence': confidence,
                    'anomaly_map': anomaly_map.squeeze().cpu().numpy(),
                    'is_normal': False
                }
        
        return None

    def detect_defects(self, image_path, grid_pattern='lines'):
        original = self.preprocess_image(image_path)
        h, w = original.shape[:2]
        
        defect_mask = np.zeros((h, w, 3), dtype=np.uint8)

        tile_size = 128
        normal_pattern = self.create_grid_pattern(tile_size, grid_size=4)

        defect_counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}

        tiles, positions = self.process_tiles(original)
        
        print(f"Processed {len(tiles)} tiles...")
        
        for tile, (x1, y1, x2, y2) in zip(tiles, positions):
            result = self.analyze_tile(tile)
            
            if result is not None:
                if result['is_normal']:
                    class_id = result['class']
                    
                    pattern_resized = normal_pattern[:tile_size, :tile_size]
                    
                    for i in range(3):
                        grid_mask = pattern_resized[:, :, i] > 0
                        defect_mask[y1:y2, x1:x2, i][grid_mask] = pattern_resized[:, :, i][grid_mask]
                    
                    grid_pixels = np.any(pattern_resized > 0, axis=2)
                    defect_counts[class_id] += np.sum(grid_pixels)
                    
                else:
                    anomaly_map = cv2.resize(result['anomaly_map'], (tile_size, tile_size))
                    defect_pixels = anomaly_map > 0.5
                    
                    if np.any(defect_pixels):
                        class_id = result['class']
                        color = self.defect_colors[class_id]
                        
                        for i in range(3):
                            defect_mask[y1:y2, x1:x2, i][defect_pixels] = color[i]
                        
                        defect_counts[class_id] += np.sum(defect_pixels)
        
        return original, defect_mask, defect_counts

    def show_results(self, image_path):
        original, defect_mask, defect_counts = self.detect_defects(image_path)
        
        original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
        
        result_image = cv2.addWeighted(original_rgb, 0.7, defect_mask, 0.3, 0)
        
        total_pixels = original.shape[0] * original.shape[1]
        defect_percentages = {}
        for class_id, count in defect_counts.items():
            defect_percentages[class_id] = (count / total_pixels) * 100
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        axes[0].imshow(original_rgb)
        axes[0].set_title('original image', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        
        axes[1].imshow(defect_mask)
        axes[1].set_title('defects map', fontsize=14, fontweight='bold')
        axes[1].axis('off')
        
        axes[2].imshow(result_image)
        axes[2].set_title('results', fontsize=14, fontweight='bold')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        normal_percentage = defect_percentages.get(0, 0)
        print(f"Normal area(green)): {normal_percentage:.2f}%")
        
        found_defects = False
        for class_id in [1, 2, 3, 4]:
            if defect_counts[class_id] > 0:
                percentage = defect_percentages[class_id]
                print(f"• {self.defect_names[class_id]}: {percentage:.2f}%")
                found_defects = True
        
        if not found_defects:
            print("defects didn't find")
        
        total_defect_percentage = sum(defect_percentages.get(i, 0) for i in [1, 2, 3, 4])
        print(f"\nTotal: {normal_percentage:.2f}% normal area, {total_defect_percentage:.2f}% defects")
        
        return result_image, defect_mask

In [ ]:
if __name__ == "__main__":
    detector = DefectDetector(r"C:\Users\Administrator\OneDrive\Desktop\Gan+Unet project\2_anomaly_detection.pth")
    result, mask = detector.show_results(r"C:\Users\Administrator\OneDrive\Desktop\diana\Gan Anomaly\article_jpg1.jpg")